# Segmentation Baseline — Abdominal Structures

Trains a lightweight **FCN decoder** on top of a **frozen encoder** (MAE, VICReg, or pretrained timm ViT)  
to segment 4 fetal abdominal structures: aorta, umbilical vein, stomach, liver.

## Workflow
1. Mount Drive & configure paths
2. Set encoder and decoder hyperparameters
3. **Train** with live loss / IoU / Dice plots
4. Resume from checkpoint (optional)
5. Evaluate and inspect per-class results

In [ ]:
!ls /content

In [ ]:
%matplotlib inline

import os, sys, json
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

# project root is two levels up from baselines/seg_abdominal/
PROJECT_ROOT = str(Path.cwd().resolve().parents[0])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
ABDOMINAL_ROOT = "/content/drive/MyDrive/SHAD/project/FetalAbdominalSegmentationDataset"

# ── Encoder ───────────────────────────────────────────────────────────────────
# Choose ONE of the three options below, comment out the others.

# Option A: pretrained timm ViT (no checkpoint needed)
ENCODER_TYPE            = "timm"
TIMM_MODEL              = "vit_small_patch16_224.dino"
PATH_ENCODER_CHECKPOINT = ""

# Option B: MAE checkpoint
# ENCODER_TYPE            = "mae"
# TIMM_MODEL              = ""
# PATH_ENCODER_CHECKPOINT = "/content/drive/MyDrive/SHAD/project/checkpoints/mae/mae_best.pt"

# Option C: VICReg checkpoint
# ENCODER_TYPE            = "vicreg"
# TIMM_MODEL              = ""
# PATH_ENCODER_CHECKPOINT = "/content/drive/MyDrive/SHAD/project/checkpoints/vicreg/v3.2/vicreg_best.pt"

RANDOM_ENCODER = False   # True → reinitialise encoder weights (ablation)

# ── Decoder ───────────────────────────────────────────────────────────────────
DECODER_HIDDEN_DIM = 256
DECODER_DEPTH      = 2

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS       = 50
BATCH_SIZE   = 16
LR           = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SPLIT    = 0.2
NUM_WORKERS  = 4
TARGET_SIZE  = 224
SEED         = 42

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/content/drive/MyDrive/SHAD/project/checkpoints/seg_abdominal"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_HEAD_CHECKPOINT = os.path.join(OUTPUT_DIR, "best_seg_head.pt")
METRICS_CSV          = os.path.join(OUTPUT_DIR, "seg_metrics.csv")
CONFIG_PATH          = os.path.join(OUTPUT_DIR, "seg_run_config.json")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Train

Plots live: **total loss**, **BCE / Dice components**, **mean IoU**, **mean Dice** (train / val).

In [ ]:
import csv, random
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

from datasets_adapters.abdominal_segmentation.abdominal_dataset import (
    AbdominalSegmentationDataset, STRUCTURE_KEYS,
)
from baselines.seg_abdominal.run import (
    FCNDecoder, SegmentationModel,
    _load_encoder, collate_fn,
    train_one_epoch, evaluate,
    seg_loss, compute_metrics,
    _save_checkpoint, NUM_CLASSES,
)

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

# ── Load encoder ──────────────────────────────────────────────────────────────
os.environ["RANDOM_ENCODER"] = "1" if RANDOM_ENCODER else "0"
encoder = _load_encoder(ENCODER_TYPE, PATH_ENCODER_CHECKPOINT, TIMM_MODEL, DEVICE)
embed_dim  = encoder.embed_dim
patch_size = encoder.patch_size
print(f"Encoder: {ENCODER_TYPE}  embed_dim={embed_dim}  patch_size={patch_size}")

# ── Dataset ───────────────────────────────────────────────────────────────────
dataset = AbdominalSegmentationDataset(
    root=ABDOMINAL_ROOT,
    load_masks=True,
    target_size=(TARGET_SIZE, TARGET_SIZE),
)
n_val   = max(1, int(len(dataset) * VAL_SPLIT))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn)
print(f"Train: {n_train}  Val: {n_val}")

# ── Model ─────────────────────────────────────────────────────────────────────
decoder = FCNDecoder(embed_dim, DECODER_HIDDEN_DIM, NUM_CLASSES, patch_size, depth=DECODER_DEPTH)
decoder.to(DEVICE)
model = SegmentationModel(encoder, decoder).to(DEVICE)
n_params = sum(p.numel() for p in decoder.parameters())
print(f"Decoder parameters: {n_params:,}")

optimizer = torch.optim.AdamW(decoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ── History tracking ─────────────────────────────────────────────────────────
history = {k: [] for k in [
    "train_loss", "train_bce", "train_dice", "train_mean_iou", "train_mean_dice",
    "val_loss",   "val_bce",   "val_dice",   "val_mean_iou",   "val_mean_dice",
]}

csv_file = open(METRICS_CSV, "w", newline="")
writer = csv.DictWriter(csv_file, fieldnames=["epoch"] + list(history.keys()))
writer.writeheader()

def update_plots(history, epochs_total):
    clear_output(wait=True)
    e = len(history["train_loss"])
    xs = range(1, e + 1)

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))

    axes[0].plot(xs, history["train_loss"], label="train", lw=1.5)
    axes[0].plot(xs, history["val_loss"],   label="val",   lw=1.5)
    axes[0].set_title("Total Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(xs, history["train_bce"],  label="bce",  lw=1.5)
    axes[1].plot(xs, history["train_dice"], label="dice", lw=1.5)
    axes[1].set_title("Loss Components (train)"); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(xs, history["train_mean_iou"], label="train", lw=1.5)
    axes[2].plot(xs, history["val_mean_iou"],   label="val",   lw=1.5)
    axes[2].set_title("Mean IoU"); axes[2].legend(); axes[2].grid(alpha=0.3)
    axes[2].set_ylim(0, 1)

    axes[3].plot(xs, history["train_mean_dice"], label="train", lw=1.5)
    axes[3].plot(xs, history["val_mean_dice"],   label="val",   lw=1.5)
    axes[3].set_title("Mean Dice"); axes[3].legend(); axes[3].grid(alpha=0.3)
    axes[3].set_ylim(0, 1)

    plt.tight_layout()
    plt.show()

    ep = len(history["train_loss"])
    print(
        f"[{ep:3d}/{epochs_total}]  "
        f"loss={history['train_loss'][-1]:.4f}/{history['val_loss'][-1]:.4f}  |  "
        f"iou={history['train_mean_iou'][-1]:.4f}/{history['val_mean_iou'][-1]:.4f}  |  "
        f"dice={history['train_mean_dice'][-1]:.4f}/{history['val_mean_dice'][-1]:.4f}"
    )

# ── Training loop ─────────────────────────────────────────────────────────────
best_iou = 0.0
hist_list = []

for epoch in range(1, EPOCHS + 1):
    tr = train_one_epoch(model, train_loader, optimizer, DEVICE)
    vl = evaluate(model, val_loader, DEVICE)

    for k in history:
        split, metric = k.split("_", 1)
        history[k].append(tr[metric] if split == "train" else vl[metric])

    row = {"epoch": epoch, **{k: f"{history[k][-1]:.6f}" for k in history}}
    writer.writerow(row); csv_file.flush()
    hist_list.append(row)

    if vl["mean_iou"] > best_iou:
        best_iou = vl["mean_iou"]
        _save_checkpoint(PATH_HEAD_CHECKPOINT, decoder, optimizer, epoch, best_iou,
                         {"encoder_type": ENCODER_TYPE, "timm_model": TIMM_MODEL,
                          "embed_dim": embed_dim, "patch_size": patch_size,
                          "decoder_hidden_dim": DECODER_HIDDEN_DIM,
                          "decoder_depth": DECODER_DEPTH, "num_classes": NUM_CLASSES},
                         hist_list)

    update_plots(history, EPOCHS)

csv_file.close()
print(f"\nDone. Best val mean IoU: {best_iou:.4f}")
print(f"Checkpoint: {PATH_HEAD_CHECKPOINT}")

## 3. Resume Training (optional)

In [ ]:
from baselines.seg_abdominal.run import _load_head_checkpoint

start_epoch, best_iou, hist_list = _load_head_checkpoint(
    PATH_HEAD_CHECKPOINT, decoder, optimizer
)
start_epoch += 1
print(f"Resumed from epoch {start_epoch - 1}, best IoU={best_iou:.4f}")

# rebuild history dict from saved list
for k in history:
    history[k] = [float(r[k]) for r in hist_list if k in r]

csv_file = open(METRICS_CSV, "a", newline="")
writer = csv.DictWriter(csv_file, fieldnames=["epoch"] + list(history.keys()))

for epoch in range(start_epoch, EPOCHS + 1):
    tr = train_one_epoch(model, train_loader, optimizer, DEVICE)
    vl = evaluate(model, val_loader, DEVICE)

    for k in history:
        split, metric = k.split("_", 1)
        history[k].append(tr[metric] if split == "train" else vl[metric])

    row = {"epoch": epoch, **{k: f"{history[k][-1]:.6f}" for k in history}}
    writer.writerow(row); csv_file.flush()
    hist_list.append(row)

    if vl["mean_iou"] > best_iou:
        best_iou = vl["mean_iou"]
        _save_checkpoint(PATH_HEAD_CHECKPOINT, decoder, optimizer, epoch, best_iou,
                         {"encoder_type": ENCODER_TYPE, "timm_model": TIMM_MODEL,
                          "embed_dim": embed_dim, "patch_size": patch_size,
                          "decoder_hidden_dim": DECODER_HIDDEN_DIM,
                          "decoder_depth": DECODER_DEPTH, "num_classes": NUM_CLASSES},
                         hist_list)

    update_plots(history, EPOCHS)

csv_file.close()
print(f"\nDone. Best val mean IoU: {best_iou:.4f}")

## 4. Evaluate — per-class IoU / Dice

In [ ]:
# Load best decoder weights for evaluation
from baselines.seg_abdominal.run import _load_head_checkpoint, compute_metrics

_load_head_checkpoint(PATH_HEAD_CHECKPOINT, decoder, optimizer=None)

all_logits, all_masks = [], []
model.decoder.eval()

with torch.no_grad():
    for batch in val_loader:
        imgs  = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        logits = model(imgs)
        if logits.shape[2:] != masks.shape[2:]:
            logits = F.interpolate(logits, size=masks.shape[2:], mode="bilinear", align_corners=False)
        all_logits.append(logits.cpu())
        all_masks.append(masks.cpu())

per_class = compute_metrics(torch.cat(all_logits), torch.cat(all_masks))

print(f"\n{'Structure':<35} {'IoU':>8} {'Dice':>8}")
print("-" * 55)
for c, key in enumerate(STRUCTURE_KEYS):
    iou  = per_class[f"iou_class{c}"]
    dice = per_class[f"dice_class{c}"]
    print(f"{key:<35} {iou:>8.4f} {dice:>8.4f}")
print("-" * 55)
print(f"{'Mean':<35} {per_class['mean_iou']:>8.4f} {per_class['mean_dice']:>8.4f}")

## 5. Visualise Predictions

In [ ]:
import math

N_SHOW = 4   # number of samples to visualise
COLORS = ["Reds", "Greens", "Blues", "Purples"]

model.decoder.eval()
samples = [val_ds[i] for i in range(N_SHOW)]
from baselines.seg_abdominal.run import _pad_mask

imgs_t  = torch.stack([s["image"] for s in samples]).to(DEVICE)
masks_t = torch.stack([_pad_mask(s["mask"], NUM_CLASSES) for s in samples]).to(DEVICE)

with torch.no_grad():
    logits = model(imgs_t)
    if logits.shape[2:] != masks_t.shape[2:]:
        logits = F.interpolate(logits, size=masks_t.shape[2:], mode="bilinear", align_corners=False)
    preds = (torch.sigmoid(logits) > 0.5).float()

imgs_np  = imgs_t.cpu().squeeze(1).numpy()    # [N, H, W]
masks_np = masks_t.cpu().numpy()              # [N, C, H, W]
preds_np = preds.cpu().numpy()                # [N, C, H, W]

fig, axes = plt.subplots(N_SHOW, NUM_CLASSES + 1, figsize=(4 * (NUM_CLASSES + 1), 4 * N_SHOW))

for i in range(N_SHOW):
    axes[i, 0].imshow(imgs_np[i], cmap="gray")
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")
    for c, (key, cmap) in enumerate(zip(STRUCTURE_KEYS, COLORS)):
        ax = axes[i, c + 1]
        ax.imshow(imgs_np[i], cmap="gray")
        ax.imshow(masks_np[i, c], cmap=cmap, alpha=0.4, vmin=0, vmax=1)
        ax.imshow(preds_np[i, c], cmap="Oranges", alpha=0.3, vmin=0, vmax=1)
        short = key.split("_")[-1]
        ax.set_title(f"{short}\nGT=red  Pred=orange", fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()